In [1]:
import dotenv
import os
from pathlib import Path

from langchain_core.messages import AIMessage,HumanMessage

from langchain_community.document_loaders.text import TextLoader

from langchain_community.document_loaders import (
    WebBaseLoader,
    PyPDFLoader,
    Docx2txtLoader
)

from langchain_community.vectorstores import Chroma

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings,ChatOpenAI

from langchain_anthropic import ChatAnthropic

from langchain_groq import ChatGroq

from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings

from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.history_aware_retriever import (
    create_history_aware_retriever
)
from langchain_huggingface import HuggingFaceEmbeddings


dotenv.load_dotenv()


C:\Users\91934\AppData\Local\Temp\ipykernel_62956\3978186228.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [3]:
doc_paths=[
    "docs/test_rag.pdf",
    "docs/test_rag.docx"
]
docs=[]
for doc_file in doc_paths:
    file_path=Path(doc_file)

    try:
        if doc_file.endswith(".pdf"):
            loader=PyPDFLoader(file_path)
        elif doc_file.endswith(".docx"):
            loader=Docx2txtLoader(file_path)
        elif doc_file.endswith(".txt"):
            loader=TextLoader(file_path)
        else:
            print(f"Document type {doc_file.type} not supported")
            continue
        docs.extend(loader.load())
    except Exception as e:
        print(f"Error Loading Document {doc_file.name}:{e}")
    finally:
        # os.remove(file_path)
        print("---")

url="https://docs.streamlit.io/develop/quick-reference/release-notes"

try:
    loader = WebBaseLoader(url)
    docs.extend(loader.load())
except Exception as e:
    print(f"Error loading document from {url}:{e}")


---
---


In [4]:
docs

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-09-15T19:40:36+02:00', 'msip_label_1cf2ba15-c468-47c8-b178-cba8acf110ec_siteid': 'eb25818e-5bd5-49bf-99de-53e3e7b42630', 'msip_label_1cf2ba15-c468-47c8-b178-cba8acf110ec_method': 'Standard', 'msip_label_1cf2ba15-c468-47c8-b178-cba8acf110ec_enabled': 'True', 'author': 'Domingo Domènech Enric (ERNI)', 'moddate': '2024-09-15T19:40:36+02:00', 'source': 'docs\\test_rag.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='My favorite food is margarita pizza. \nThere are 47588 bottles in the truck.'),
 Document(metadata={'source': 'docs\\test_rag.docx'}, page_content='My favorite food is margarita pizza.\n\nThere are 47588 bottles in the truck.'),
 Document(metadata={'source': 'https://docs.streamlit.io/develop/quick-reference/release-notes', 'title': 'Release notes - Streamlit Docs', 'description': 'A changelog of highlights and fixes for 

In [5]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=5000,
    chunk_overlap=1000
)

document_chunks=text_splitter.split_documents(docs)


In [15]:
vector_db=Chroma.from_documents(
    documents=document_chunks,
    embedding=GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001"
    )
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
def _get_context_retriever_chain(vector_db,llm):
    retriever=vector_db.as_retriever()
    prompt=ChatPromptTemplate.from_messages(
        [
            MessagesPlaceholder(variable_name="messages"),
            ("user","{input}"),
            ("user","Given the above conversation, generate a search query to look up inorder to get relevant information, focusing on the most recet messages.")
        ]
    )
    retriever_chain=create_history_aware_retriever(llm,retriever,prompt)

    return retriever_chain

In [17]:
def get_conversational_rag_chain(llm):
    retriever_chain = _get_context_retriever_chain(vector_db, llm)

    prompt = ChatPromptTemplate.from_messages([
        ("system",
        """You are a helpful assistant. You will have to answer to user's queries.
        You will have some context to help with your answers, but now always would be completely related or helpful.
        You can also use your knowledge to assist answering the user's queries.\n
        {context}"""),
        MessagesPlaceholder(variable_name="messages"),
        ("user", "{input}"),
    ])
    stuff_documents_chain = create_stuff_documents_chain(llm, prompt)

    return create_retrieval_chain(retriever_chain, stuff_documents_chain)

In [23]:
# Gemini LLM
llm_stream_gemini = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0.3,
)
llm_stream_openrouter= ChatOpenAI(
    model="google/gemini-2.5-flash",
    base_url="https://openrouter.ai/api/v1",
    api_key="YOUR_OPENROUTER_API_KEY",
    temperature=0.3,
)
llm_stream=llm_stream_gemini
messages = [
    HumanMessage(content="Hi"),
    AIMessage(content="Hi there! How can I assist you today?"),
    HumanMessage(content="What is the latest version of Streamlit? and I would like to Know the highlights of the latest version?")
]

conversation_rag_chain = get_conversational_rag_chain(llm_stream)

response_message = "*(RAG Response)*\n"

for chunk in conversation_rag_chain.pick("answer").stream({
    "messages": messages[:-1],
    "input": messages[-1].content
}):
    response_message += chunk
    print(chunk, end="", flush=True)

messages.append(
    AIMessage(content=response_message)
)

The latest version of Streamlit is **Version 1.60.0**, released on July 21, 2026.

**Highlights of this version include:**

*   **App-wide Data Export Control:** You can now disable bulk data export across your entire app using the new `client.disableDataExport` configuration option.
*   **Hidden Export Buttons:** When the above option is enabled, the CSV export button is hidden for `st.dataframe`, `st.data_editor`, and chart table views.
*   **Clipboard Protection:** Clipboard copying is disabled for read-only dataframes when the data export is restricted.

Additionally, this release includes several security hardenings, such as rejecting host messages from child iframes to prevent origin spoofing and capping client-supplied query strings to guard against resource exhaustion.

In [14]:
from google import genai
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

for model in client.models.list():
    if "embedContent" in getattr(model, "supported_actions", []):
        print(model.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [20]:
from google import genai

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-